# TWFE / OLS benchmark — B2_sweep (linearity_degree=1)

**Workstream B2 · canonical DiD (selection on unobservables)**

Plain two-way fixed-effects OLS benchmark for the `B2_sweep` scenario at
`linearity_degree=1`: the event-study path ATT(k) (k ≥ 0) and the overall
static ATT, each with cluster-robust (by unit) standard errors. This is the foil
DiD-BCF is compared against -- and, under staggered dynamic effects, the
estimator Goodman-Bacon (2021) shows is contaminated. Pure numpy/pandas (no
stochtree), so it runs on a laptop with parallelisation.

> **Colab:** upload just this notebook and *Run all* — the setup cell clones the
> engine automatically (no extra installs needed).

In [ ]:
import os, sys

# --- Locate the DiD-BCF engine ------------------------------------------------
# So you can upload just THIS notebook to Colab and Run all. Resolution order:
#   1. `did_bcf_revision` already importable;
#   2. running inside a repo checkout (the parent folder holds the package);
#   3. otherwise clone https://github.com/hugogobato/DiD-BCF and use it.
REPO_URL = "https://github.com/hugogobato/DiD-BCF.git"
ENGINE_SUBDIR = os.path.join("DiD-BCF", "Simulation_Studies_Revision")

def _locate_root():
    try:
        import did_bcf_revision  # noqa: F401
        return os.path.dirname(os.path.dirname(did_bcf_revision.__file__))
    except Exception:
        pass
    parent = os.path.abspath(os.path.join(os.getcwd(), ".."))
    if os.path.isdir(os.path.join(parent, "did_bcf_revision")):
        return parent
    if not os.path.isdir("DiD-BCF"):
        import subprocess
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL], check=True)
    return os.path.abspath(ENGINE_SUBDIR)

ROOT = _locate_root()
sys.path.insert(0, ROOT)
print("Using DiD-BCF engine at:", ROOT)

from did_bcf_revision.twfe_runner import run_twfe_named
from did_bcf_revision.metrics import compute_metrics, surface_metrics

In [ ]:
REPS = 100
JOBS = 1        # raise to parallelise replications on your PC

summaries = run_twfe_named("B2_sweep", linearity_degree=1, reps=REPS, jobs=JOBS)
summaries.head()

In [ ]:
# Decomposed metrics (incl. MAE/MAPE, calibration ratio, MC SEs) and the
# CATT-surface RMSE/MAE/MAPE (TWFE's event-study coef broadcast to each obs --
# where heterogeneity-blind TWFE pays its price).
display(compute_metrics(summaries))
surface_metrics(summaries)